In [ ]:
from pathlib import Path
import os
import sys

try:
    import kaggle  # noqa: F401
except Exception:
    %pip -q install kaggle

PROJECT_ROOT = Path("/content/Sleep-Stage")
LOCAL_ROOT = Path("/Users/temicide/Documents/5_domain_final/Sleep-Stage")
if LOCAL_ROOT.exists():
    PROJECT_ROOT = LOCAL_ROOT

src_path = PROJECT_ROOT / "src"
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

# Colab input path: /content/input
CONTENT_ROOT = Path("/content")
INPUT_ROOT = CONTENT_ROOT / "input"
WORKING_DIR = CONTENT_ROOT / "working"
SUBMISSION_PATH = CONTENT_ROOT / "submission.csv"
INPUT_ROOT.mkdir(parents=True, exist_ok=True)
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print(f"Using project root: {PROJECT_ROOT}")


In [ ]:
from pathlib import Path
import os

uploaded_kaggle_json = None
if not (Path.home() / ".kaggle" / "kaggle.json").exists():
    if not (os.environ.get("KAGGLE_USERNAME") and os.environ.get("KAGGLE_KEY")):
        from google.colab import files
        uploaded = files.upload()
        if "kaggle.json" in uploaded:
            uploaded_kaggle_json = Path("kaggle.json")
        else:
            raise RuntimeError("Upload kaggle.json, or set KAGGLE_USERNAME and KAGGLE_KEY in Colab secrets.")

from sleep_stage.kaggle_download import configure_kaggle_credentials

credential_path = configure_kaggle_credentials(uploaded_kaggle_json)
if credential_path is None:
    raise RuntimeError("Kaggle credentials are not configured.")
print("Kaggle credentials configured.")


In [ ]:
from sleep_stage.config import COMPETITION_SLUG
from sleep_stage.kaggle_download import download_and_extract_competition

# Downloads with: kaggle competitions download
competition_root = download_and_extract_competition(INPUT_ROOT, COMPETITION_SLUG)
print(f"Competition data ready: {competition_root}")


In [ ]:
from sleep_stage.config import build_paths
from sleep_stage.pipeline import load_or_build_features, run_experiments

paths = build_paths(
    project_root=PROJECT_ROOT,
    competition_root=competition_root,
    output_path=SUBMISSION_PATH,
    working_dir=WORKING_DIR,
)
RUN_TABULAR_CV = os.environ.get("RUN_TABULAR_CV", "0") == "1"
if RUN_TABULAR_CV:
    train_table, test_table = load_or_build_features(paths, force=False)
    print("train_table", train_table.shape)
    print("test_table", test_table.shape)
    print(train_table["label"].value_counts().to_dict())
    results = run_experiments(paths=paths, force_features=False)
    print(results[["experiment", "model_name", "n_features", "weighted_f1_mean", "weighted_f1_std"]].to_string(index=False))
else:
    print("Skipping tabular CV. Set RUN_TABULAR_CV=1 to run the sklearn baseline experiments.")


In [ ]:
try:
    import torch
except Exception:
    %pip -q install torch
    import torch

from sleep_stage.deep import DeepTrainingConfig, configure_h100_runtime, run_deep_grouped_cv

configure_h100_runtime()
RUN_H100_DEEP = os.environ.get("RUN_H100_DEEP", "1") == "1" and torch.cuda.is_available()
RUN_DEEP_CV = os.environ.get("RUN_DEEP_CV", "0") == "1"
DEEP_EPOCHS = int(os.environ.get("DEEP_EPOCHS", "18"))

deep_config = DeepTrainingConfig(
    context_epochs=int(os.environ.get("DEEP_CONTEXT_EPOCHS", "11")),
    batch_size=int(os.environ.get("DEEP_BATCH_SIZE", "512")),
    epochs=DEEP_EPOCHS,
    width=int(os.environ.get("DEEP_WIDTH", "256")),
    transformer_layers=int(os.environ.get("DEEP_LAYERS", "4")),
    transformer_heads=int(os.environ.get("DEEP_HEADS", "8")),
    num_workers=int(os.environ.get("DEEP_NUM_WORKERS", "2")),
)

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
print("RUN_H100_DEEP:", RUN_H100_DEEP)
print(deep_config)

if RUN_H100_DEEP and RUN_DEEP_CV:
    # Set RUN_DEEP_CV=1 for serious model selection. By default the notebook trains
    # the final deep model directly to produce a submission within one Colab session.
    cv_summary = run_deep_grouped_cv(paths, config=deep_config, n_splits=5)
    print(cv_summary.to_string(index=False))


In [ ]:
from pathlib import Path
import pandas as pd

from sleep_stage.data import validate_submission
from sleep_stage.deep import train_final_deep_submission
from sleep_stage.pipeline import generate_submission

if RUN_H100_DEEP:
    submission_path = train_final_deep_submission(paths, config=deep_config, force_raw=False)
else:
    submission_path = generate_submission(paths)
validation = validate_submission(submission_path, paths.sample_submission)
submission = pd.read_csv(submission_path)

print(validation)
print(submission.head().to_string(index=False))
print(submission["labels"].value_counts().to_string())
print(f"Submission CSV ready: {submission_path}")
assert submission_path == Path("/content/submission.csv") or LOCAL_ROOT.exists()
assert len(submission) == 7832
